
# 1. Installs
Installing the required external libraries for cloud data extraction.

In [0]:
# Installing Kaggle API
%pip install kaggle


# 2. Imports & APIs
Importing built-in modules to set up authentication credentials securely before initializing the Kaggle API.

In [0]:
import os

# The dbutils retrieve the keys directly from the secure vault at Databricks.
os.environ['KAGGLE_USERNAME'] = dbutils.secrets.get(scope="kaggle_scope", key="username")
os.environ['KAGGLE_KEY'] = dbutils.secrets.get(scope="kaggle_scope", key="key")

import kaggle


# 3. Environment Variables
Defining the core paths and schemas for the Bronze layer ingestion.

In [0]:
# Defining the secure landing zone path (Unity Catalog Volume)
volume_path = "/Volumes/workspace/bronze_marketing_project/raw_data"


# 4. Data Extraction
Downloading and extracting the Olist E-commerce dataset directly from Kaggle into the Databricks Volume.

In [0]:
# Downloading the dataset directly to the landing zone
kaggle.api.dataset_download_files('olistbr/brazilian-ecommerce', path=volume_path, unzip=True)


# 5. Batch Ingestion (Bronze)
Iterating through the raw CSV files in the Volume and writing them as structured Delta Tables into the Bronze schema.

In [0]:
# Listing all files in the secure volume
files = dbutils.fs.ls(volume_path)

for file_info in files:
    if file_info.name.endswith(".csv"):
        file_name = file_info.name
        
        # Standardizing table names (e.g., from 'olist_orders_dataset.csv' to 'bronze_orders')
        table_name = "bronze_" + file_name.replace("olist_", "").replace("_dataset", "").replace(".csv", "")
        full_table_path = f"workspace.bronze_marketing_project.{table_name}"
        
        print(f"Processing: {file_name} -> {full_table_path}")
        
        # Reading raw CSV dynamically
        df = (spark.read
              .format("csv")
              .option("header", "true")
              .option("inferSchema", "true")
              .load(file_info.path)
        )
        
        # Writing as Delta Table (Idempotent operation)
        (df.write
          .format("delta")
          .mode("overwrite")
          .saveAsTable(full_table_path)
        )